In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
files = {
    "Wild vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Wild_vs_GF_results.csv",
    "Conventionalized vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Conventionalized_vs_GF_results.csv",
    "Curvibacter vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Cvbct_vs_GF_results.csv",
    "3OHC12 vs. Control": "../results/deseq2_rsem/tables/HydraAHL_3OHC12_vs_control_results.csv",
}

lfc_tables = []

for experiment, filename in files.items():
    df = pd.read_csv(filename)
    df = df[df["padj"] <= 0.05]
    #df = df[df["log2FoldChange"] < 0.0]
    df = df[["gene", "log2FoldChange"]].copy()
    df = df.rename(columns={"log2FoldChange": experiment})

    lfc_tables.append(df)

lfc = lfc_tables[0]

for df in lfc_tables[1:]:
    lfc = lfc.merge(df, on="gene", how="inner")

lfc = lfc.set_index("gene")

lfc = lfc.replace([np.inf, -np.inf], np.nan)
lfc = lfc.dropna()

lfc = lfc.loc[lfc.std(axis=1) > 0]

X = lfc.T

X_centered = X - X.mean(axis=0)

pca = PCA(n_components=2)

scores = pca.fit_transform(X_centered)

pca_df = pd.DataFrame(
    scores,
    index=X.index,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(7, 6))

plt.scatter(pca_df["PC1"], pca_df["PC2"])

for experiment, row in pca_df.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        experiment
    )

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title("PCA of transcriptional responses")
plt.tight_layout()
plt.show()

correlation = lfc.corr(method="pearson")
column_header = [ "Wild vs. GF","Conv. vs. GF","Curvi. vs. GF","3OHC12 vs. EtAc"]
correlation.columns = column_header
correlation.index = column_header

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(correlation.values,vmin=0,
    vmax=1,cmap="YlGnBu")

ax.set_xticks(range(len(correlation.columns)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right", fontsize=18)

ax.set_yticks(range(len(correlation.index)))
ax.set_yticklabels(correlation.index, fontsize=18)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Pearson correlation", fontsize=18, labelpad=20)
cbar.ax.tick_params(labelsize=16)
#plt.title("Similarity of transcriptional responses")
#plt.tight_layout()
plt.savefig("../results/figures/correlation_heatmap_pearson.svg")

In [ ]:
files = {
    "Wild vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Wild_vs_GF_results.csv",
    "Conventionalized vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Conventionalized_vs_GF_results.csv",
    "Curvibacter vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Cvbct_vs_GF_results.csv",
    "3OHC12 vs. Control": "../results/deseq2_rsem/tables/HydraAHL_3OHC12_vs_control_results.csv",
}

lfc_tables = []

for experiment, filename in files.items():
    df = pd.read_csv(filename)
    df = df[df["padj"] <= 0.05]
    df = df[df["log2FoldChange"] > 0.0]
    df = df[["gene", "log2FoldChange"]].copy()
    df = df.rename(columns={"log2FoldChange": experiment})

    lfc_tables.append(df)

lfc = lfc_tables[0]

for df in lfc_tables[1:]:
    lfc = lfc.merge(df, on="gene", how="inner")

lfc = lfc.set_index("gene")

lfc = lfc.replace([np.inf, -np.inf], np.nan)
lfc = lfc.dropna()

lfc = lfc.loc[lfc.std(axis=1) > 0]

X = lfc.T

X_centered = X - X.mean(axis=0)

pca = PCA(n_components=2)

scores = pca.fit_transform(X_centered)

pca_df = pd.DataFrame(
    scores,
    index=X.index,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(7, 6))

plt.scatter(pca_df["PC1"], pca_df["PC2"])

for experiment, row in pca_df.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        experiment
    )

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title("PCA of transcriptional responses")
plt.tight_layout()
plt.show()

correlation = lfc.corr(method="pearson")
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(correlation.values,vmin=0,
    vmax=1)

ax.set_xticks(range(len(correlation.columns)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right")

ax.set_yticks(range(len(correlation.index)))
ax.set_yticklabels(correlation.index)

fig.colorbar(im, ax=ax, label="Pearson correlation")

plt.title("Similarity of transcriptional responses")
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv(files["Curvibacter vs. GF"])
df = df[df["padj"] <= 0.05]
df = df[df["log2FoldChange"] > 0.0]
df = df[["gene", "log2FoldChange"]].copy()
df = df.rename(columns={"log2FoldChange": "Curvibacter"})

In [ ]:
df1 = pd.read_csv(files["3OHC12 vs. Control"])
df1 = df1[df1["padj"] <= 0.05]
df1 = df1[df1["log2FoldChange"] > 0.0]
df1 = df1[["gene", "log2FoldChange"]].copy()
df1 = df1.rename(columns={"log2FoldChange": "HydraAHL"})

In [ ]:
df.merge(df1, on="gene")

In [ ]:
lfc_tables

In [ ]:
files = {
    "Wild vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Wild_vs_GF_results.csv",
    "Conventionalized vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Conventionalized_vs_GF_results.csv",
    "Curvibacter vs. GF": "../results/deseq2_rsem/tables/HydraRecolonization_Cvbct_vs_GF_results.csv",
    "3OHC12 vs. Control": "../results/deseq2_rsem/tables/HydraAHL_3OHC12_vs_control_results.csv",
}

lfc_tables = []

for experiment, filename in files.items():
    df = pd.read_csv(filename)
    df = df[df["padj"] <= 0.05]
    df = df[df["log2FoldChange"] < 0.0]
    df = df[["gene", "log2FoldChange"]].copy()
    df = df.rename(columns={"log2FoldChange": experiment})

    lfc_tables.append(df)

lfc = lfc_tables[0]

for df in lfc_tables[1:]:
    lfc = lfc.merge(df, on="gene", how="inner")

lfc = lfc.set_index("gene")

lfc = lfc.replace([np.inf, -np.inf], np.nan)
lfc = lfc.dropna()

lfc = lfc.loc[lfc.std(axis=1) > 0]

X = lfc.T

X_centered = X - X.mean(axis=0)

pca = PCA(n_components=2)

scores = pca.fit_transform(X_centered)

pca_df = pd.DataFrame(
    scores,
    index=X.index,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(7, 6))

plt.scatter(pca_df["PC1"], pca_df["PC2"])

for experiment, row in pca_df.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        experiment
    )

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)
plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title("PCA of transcriptional responses")
plt.tight_layout()
plt.show()

correlation = lfc.corr(method="pearson")
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(correlation.values,vmin=0,
    vmax=1)

ax.set_xticks(range(len(correlation.columns)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right")

ax.set_yticks(range(len(correlation.index)))
ax.set_yticklabels(correlation.index)

fig.colorbar(im, ax=ax, label="Pearson correlation")

plt.title("Similarity of transcriptional responses")
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv(
    "../results/deseq2_rsem/combined_analysis/combined_sample_PCoA_coordinates.csv"
)

df["condition"] = df["condition"].apply(lambda x: "Curvi." if x == "Cvbct" else x)
df["condition"] = df["condition"].apply(lambda x: "EtAc" if x == "control" else x)
df["condition"] = df["condition"].apply(lambda x: "Conv." if x == "Conventionalized" else x)
#df["condition"] = df["condition"].apply(lambda x: "Curvi." if x == "Curvibacter" else x)

print(df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for condition, group in df.groupby("condition"):
    ax.scatter(
        group["PCoA1"],
        group["PCoA2"],
        label=condition,
        s=150
    )

#for _, row in df.iterrows():
#    ax.text(
#        row["PCoA1"],
#        row["PCoA2"],
#       row["sample_name"],
#        fontsize=8
#    )

ax.set_xlabel("PCoA1 - 60.3%",fontsize=16, labelpad=15)
ax.set_ylabel("PCoA2 - 8.7%",fontsize=16, labelpad=15)
#ax.set_title("Combined sample-level PCoA")
ax.legend(fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.savefig("../results/deseq2_rsem/combined_analysis/pcoa_python_plot.svg",dpi=400)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

condition_order = ["Wild", "Conv.", "Curvi.", "GF", "3OHC12", "EtAc"]

for condition in condition_order:
    group = df[df["condition"] == condition]

    ax.scatter(
        group["PCoA1"],
        group["PCoA2"],
        label=condition,
        s=150
    )

ax.set_xlabel("PCoA1 - 60.3%", fontsize=18, labelpad=15)
ax.set_ylabel("PCoA2 - 8.7%", fontsize=18, labelpad=15)

ax.legend(fontsize=16,loc="upper left")

plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()

plt.savefig("../results/deseq2_rsem/combined_analysis/pcoa_python_plot.svg",dpi=400)